In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
# from huggingface_hub import snapshot_download

# snapshot_download(
#     repo_id="SparkAudio/voxbox", 
#     repo_type="dataset", 
#     local_dir="./magicdata",
#     allow_patterns="audios/magicdata/*.tar.gz"
# )

In [3]:
# import tarfile

# def loop(files):
#     files, _ = files
#     for f in tqdm(files):
#         try:
#             with tarfile.open(f, "r:gz") as tar:
#                 tar.extractall(path='magicdata')
#             os.remove(f)
#         except Exception as e:
#             print(e)

In [4]:
# files = glob('magicdata/audios/magicdata/*.tar.gz')
# multiprocessing(files, loop, len(files), returned = False)

In [5]:
# !wget https://huggingface.co/datasets/SparkAudio/voxbox/resolve/main/metadata/magicdata.jsonl
# !wget https://huggingface.co/datasets/SparkAudio/voxbox/resolve/main/speaker_ids/magicdata.txt

In [6]:
with open('magicdata.txt') as fopen:
    speakers = fopen.read().split('\n')
speakers = [s for s in speakers if '\t' in s]
speakers_mapping = {}
for s in speakers:
    l, r = s.split('\t')
    speakers_mapping[l] = r
len(speakers_mapping)

609488

In [7]:
rows = []
with open('magicdata.jsonl') as fopen:
    for l in fopen:
        l = json.loads(l)
        rows.append(l)
len(rows)

609474

In [8]:
len(glob('magicdata_audio/*.mp3'))

609286

In [11]:
def loop(rows):
    rows, _ = rows
    data = []
    base = 'magicdata_audio'
    os.makedirs(base, exist_ok=True)
    for row in tqdm(rows):
        try:
        
            if row['index'] not in speakers_mapping:
                continue
                
            speaker = speakers_mapping[row['index']]
    
            if not os.path.exists(row['wav_path']):
                continue
    
            t = row['text'].strip()
            if len(t) < 2:
                continue
    
            audio_filename = os.path.join(base, row['wav_path'].replace('.wav', '.mp3').replace('/', '-'))
            if not os.path.exists(audio_filename):
                continue
            # audio_np, sr = sf.read(row['wav_path'])
            # if audio_np.ndim > 1:
            #     audio_np = audio_np.mean(axis=1)
            # if audio_np.shape[0] < 10000:
            #     continue
            # sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{speaker}"
            })
        except Exception as e:
            pass
            
    return data

In [12]:
data = loop((rows, 0))

100%|██████████| 609474/609474 [00:04<00:00, 152358.89it/s]


In [13]:
# data = multiprocessing(rows, loop, cores = 30)

In [16]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'magicdata_audio/magicdata-magicdata_0000000000.mp3',
 'text': '高德地图',
 'speaker': 'magicdata_audio_37_5622'}

In [17]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'magicdata')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  7.74ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1): 100%|█████████▉| 17.0MB / 17.0MB, 1.89MB/s  
Processing Files (1 / 1): 100%|██████████| 17.0MB / 17.0MB, 1.79MB/s  
Processing Files (1 / 1): 100%|██████████| 17.0MB / 17.0MB, 1.77MB/s  
New Data Upload: 100%|██████████| 17.0MB / 17.0MB, 1.77MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:10<00:00, 10.33s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/a6174f6657bcdbdc6540c6c87b56361620b253ba', commit_message='Upload dataset', commit_description='', oid='a6174f6657bcdbdc6540c6c87b56361620b253ba', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [18]:
audio_files = [d['audio_filename'] for d in data]

with open('magicdata-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [22]:
# !zip -rq magicdata_audio.zip magicdata_audio

In [21]:
# !hf upload malaysia-ai/Multilingual-TTS magicdata_audio.zip --repo-type=dataset

In [28]:
# !zip -rq magicdata_audio_neucodec.zip magicdata_audio_neucodec

In [27]:
# !hf upload malaysia-ai/Multilingual-TTS magicdata_audio_neucodec.zip --repo-type=dataset